<a href="https://colab.research.google.com/github/paulogalinaris/MicroMouse/blob/main/C%C3%B3pia_de_MicroMouse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Definição da matriz da pista
LINHAS, COLUNAS = 6, 6
OBJETIVO = (0, 5)
matriz = [[-1 for _ in range(COLUNAS)] for _ in range(LINHAS)]
paredes = [[ [False, False, False, False] for _ in range(COLUNAS)] for _ in range(LINHAS)] # Ordem: Norte, Leste, Sul, Oeste
# parede_oeste = paredes[0][0][3] é o Oeste da posição 0,0. Ou seja, esquerda da posição 0,0

# Oritentação, posição, chegada
victory_coltec = 0
erro_saida = 0
orientacao = 0 # 0 = Norte, 1 = Leste, 2 = Sul, 3 = Oeste
posicao_atual = (5, 0)
linha_atual = posicao_atual[0]
coluna_atual = posicao_atual[1]
movimentos = [(-1, 0), (0, 1), (1, 0), (0, -1)] # Norte, Leste, Sul, Oeste

# Variáveis do mapeamento
number_celula = 0
fila = []
proxima_fila = []
fila.append((OBJETIVO[0], OBJETIVO[1]))

# Função para registrar a parede em que a passagem está bloquada
def registrar_parede(direcao_relativa):
  direcao_absoluta = (orientacao + direcao_relativa) % 4
  # 1. Marca a parede na célula atual
  paredes[linha_atual][coluna_atual][direcao_absoluta] = True
  # 2. Descobre a célula vizinha usando o vetor de movimento
  deslocamento_linha, deslocamento_coluna = movimentos[direcao_absoluta]
  linha_vizinho = linha_atual + deslocamento_linha
  coluna_vizinho = coluna_atual + deslocamento_coluna
  # 3. Se o vizinho estiver dentro do labirinto, marca a parede oposta nele
  if 0 <= linha_vizinho < LINHAS and 0 <= coluna_vizinho < COLUNAS:
    dir_oposta = (direcao_absoluta + 2) % 4
    paredes[linha_vizinho][coluna_vizinho][dir_oposta] = True

# VOID LOOP -------------------------------------------------------------
# Repete o Loop até chegar à posição final
while not victory_coltec and not erro_saida:

  # Tela inicial
  print(f"\n--- Posição atual: {posicao_atual} ---")
  if orientacao == 0:
    print("Orientação: Norte")
  elif orientacao == 1:
    print("Orientação: Leste")
  elif orientacao == 2:
    print("Orientação: Sul")
  elif orientacao == 3:
    print("Orientação: Oeste")

  # Usamos input() para você digitar o valor no console (0 ou 1)
  parede_esquerda = int(input("Há parede na esquerda? (1 para Sim, 0 para Não): "))
  parede_frente = int(input("Há parede na frente? (1 para Sim, 0 para Não): "))
  parede_direita = int(input("Há parede na direita? (1 para Sim, 0 para Não): "))

  # Atribuindo as paredes
  if(parede_frente):
    registrar_parede(0) # 0 = norte

  if(parede_direita):
    registrar_parede(1) # 1 = leste

  if(parede_esquerda):
    registrar_parede(3) # 3 = oeste

  # Reset da matriz de números antes de iniciar o Flood Fill
  matriz = [[-1 for _ in range(COLUNAS)] for _ in range(LINHAS)]
  fila = [OBJETIVO]
  proxima_fila = []
  number_celula = 0

  # Atribuição de números a cada célula
  while fila:
    linha0_fila, coluna0_fila = fila.pop(0)
    matriz[linha0_fila][coluna0_fila] = number_celula

    if linha0_fila - 1 >= 0 and not paredes[linha0_fila][coluna0_fila][0] and matriz[linha0_fila - 1][coluna0_fila] == -1: # Norte
      proxima_fila.append((linha0_fila - 1, coluna0_fila))
    if linha0_fila + 1 < LINHAS and not paredes[linha0_fila][coluna0_fila][2] and matriz[linha0_fila + 1][coluna0_fila] == -1: # Sul
      proxima_fila.append((linha0_fila + 1, coluna0_fila))
    if coluna0_fila + 1 < COLUNAS and not paredes[linha0_fila][coluna0_fila][1] and matriz[linha0_fila][coluna0_fila + 1] == -1: # Leste
      proxima_fila.append((linha0_fila, coluna0_fila + 1))
    if coluna0_fila - 1 >= 0 and not paredes[linha0_fila][coluna0_fila][3] and matriz[linha0_fila][coluna0_fila - 1] == -1: # Oeste
      proxima_fila.append((linha0_fila, coluna0_fila - 1))

    # Reinicia as filas
    if not fila:
      number_celula += 1
      fila = proxima_fila.copy()
      proxima_fila.clear()

  # Print do Mapa
  for linha in matriz:
    print(linha)
  #print(fila)
  #print(proxima_fila)

  # Análise dos caminhos
  numero_atual = matriz[linha_atual][coluna_atual]
  if(linha_atual - 1 >= 0) and not paredes[linha_atual][coluna_atual][0]: # Norte
    numero_norte = matriz[linha_atual - 1][coluna_atual]
  else:
    numero_norte = 256
  if(coluna_atual + 1 < COLUNAS) and not paredes[linha_atual][coluna_atual][1]: # Leste
    numero_leste = matriz[linha_atual][coluna_atual + 1]
  else:
    numero_leste = 256
  if(linha_atual + 1 < LINHAS) and not paredes[linha_atual][coluna_atual][2]: # Sul
    numero_sul= matriz[linha_atual + 1][coluna_atual]
  else:
    numero_sul = 256
  if(coluna_atual - 1 >= 0) and not paredes[linha_atual][coluna_atual][3]: # Oeste
    numero_oeste = matriz[linha_atual][coluna_atual - 1]
  else:
    numero_oeste = 256

  # Encontra a direção mais curta
  numeros_direcoes = [numero_norte, numero_leste, numero_sul, numero_oeste]
  menor_numero = min(numeros_direcoes)

  # Caso não haja nenhuma saída disponível
  if (numero_norte == 256 and numero_leste == 256 and numero_sul == 256 and numero_oeste == 256):
    print("Erro. Todas as saídas estão bloqueadas")
    erro_saida = 1

  # Caso haja saídas
  else:
    # Encontra TODAS as direções que possuem esse menor número
    # (Ex: se norte e leste empatarem, vai retornar [0, 1])
    melhores_direcoes = [i for i, valor in enumerate(numeros_direcoes) if valor == menor_numero]
    # Se houver empate, você segue pela orientação que já estava antes
    if orientacao in melhores_direcoes:
      escolha_direcao = orientacao
    else:
      # Se não puder seguir em frente, pega a primeira opção válida da lista de empates
      escolha_direcao = melhores_direcoes[0]

    # Movimento e Giro
    orientacao = escolha_direcao
    linha_atual += movimentos[escolha_direcao][0]
    coluna_atual += movimentos[escolha_direcao][1]
    posicao_atual = (linha_atual, coluna_atual)
    print(f"Robô moveu-se para: ({linha_atual}, {coluna_atual}) virado para a direção {orientacao}")





  # Condição de vitória temporária para teste (quando chegar no objetivo)
  if posicao_atual[0] == OBJETIVO[0] and posicao_atual[1] == OBJETIVO[1]:
    victory_coltec = True


--- Posição atual: (5, 0) ---
Orientação: Norte
Há parede na esquerda? (1 para Sim, 0 para Não): 1
Há parede na frente? (1 para Sim, 0 para Não): 0
Há parede na direita? (1 para Sim, 0 para Não): 1
[5, 4, 3, 2, 1, 0]
[6, 5, 4, 3, 2, 1]
[7, 6, 5, 4, 3, 2]
[8, 7, 6, 5, 4, 3]
[9, 8, 7, 6, 5, 4]
[10, 9, 8, 7, 6, 5]
Robô moveu-se para: (4, 0) virado para a direção 0

--- Posição atual: (4, 0) ---
Orientação: Norte
Há parede na esquerda? (1 para Sim, 0 para Não): 1
Há parede na frente? (1 para Sim, 0 para Não): 0
Há parede na direita? (1 para Sim, 0 para Não): 0
[5, 4, 3, 2, 1, 0]
[6, 5, 4, 3, 2, 1]
[7, 6, 5, 4, 3, 2]
[8, 7, 6, 5, 4, 3]
[9, 8, 7, 6, 5, 4]
[10, 9, 8, 7, 6, 5]
Robô moveu-se para: (3, 0) virado para a direção 0

--- Posição atual: (3, 0) ---
Orientação: Norte
Há parede na esquerda? (1 para Sim, 0 para Não): 1
Há parede na frente? (1 para Sim, 0 para Não): 0
Há parede na direita? (1 para Sim, 0 para Não): 0
[5, 4, 3, 2, 1, 0]
[6, 5, 4, 3, 2, 1]
[7, 6, 5, 4, 3, 2]
[8, 7, 6, 5, 4